In [ ]:
import scanpy as sc
import torch
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import scipy
import pandas as pd
import decoupler as dc
import anndata
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import GroupShuffleSplit
from sklearn.utils import shuffle
import os

sc.set_figure_params(figsize=(4, 4))

date = "DATE"

In [ ]:
base_path = "/home/EOCRC_atlas/"

In [ ]:
# load in Tier2 annotated adata 
adata = sc.read_h5ad(os.path.join(base_path, 'data/all_samples_processed_withTier2annotation.h5ad'))

In [ ]:
import os
samples = os.listdir(os.path.join(base_path, 'results/DATE_EOCRC_inferCNV/outputs_for_VM'))

In [ ]:
# load in inferCNV results 
dfs = []
samples.remove("COLFR0366_T1")# remove sample that failed inferCNV 
samples.remove("COLFR0034_T1")# remove sample that didn't have the right HMM.dat file - look into why it wasn't generated 

for d in samples: 
    df = pd.read_csv(os.path.join(base_path, f'results/DATE_EOCRC_inferCNV/outputs_for_VM/{d}/infercnv/HMM_CNV_predictions.HMMi6.leiden.hmm_mode-subclusters.Pnorm_0.5.pred_cnv_genes.dat'), sep='\t')
        
    # remove the non-epithelial data
    df = df[~df['cell_group_name'].str.contains('Non-epithelial', case=False, na=False)]
    
    # add sample ID to the df 
    df['cell_group_name'] = d+'_'+df['cell_group_name']

    # look up sample age information 
    decade = np.unique(adata.obs['Decade'][adata.obs['FRID']==d])
    cohort = np.unique(adata.obs['Cohort'][adata.obs['FRID']==d])
    msi = np.unique(adata.obs['MSI_v2'][adata.obs['FRID']==d])
    side = np.unique(adata.obs['Sidedness'][adata.obs['FRID']==d])
    stage = np.unique(adata.obs['Overall_Stage'][adata.obs['FRID']==d])
    sex = np.unique(adata.obs['Sex'][adata.obs['FRID']==d])
    treat = np.unique(adata.obs['Therapy_v2'][adata.obs['FRID']==d])
    age = np.unique(adata.obs['Age'][adata.obs['FRID']==d])

    mean_counts = adata.obs['total_counts'][adata.obs['FRID'] == d].mean()
    epi_count = (
        (adata.obs['FRID'] == d) & 
        (adata.obs['Annotation_Tier1'] == 'Epithelial') & 
        (adata.obs['Annotation_Tier2'] != 'Mixed - epithelial')
    ).sum()
    
    df['Decade']=decade[0]
    df['Cohort']=cohort[0]
    df['MSI']=msi[0]
    df['Side']=side[0]
    df['Stage']=stage[0]
    df['Sex']=sex[0]
    df['Treatment']=treat[0]
    df['Age']=age[0]
    df['Mean_Counts']=mean_counts
    df['Total_Epi']=epi_count


    dfs.append(df)

In [ ]:
# Create an empty list to store the intermediate results
chunk_size = 10  # number of DataFrames per chunk
chunks = []
for i in range(0, len(dfs), chunk_size):
    chunk = pd.concat(dfs[i:i+chunk_size])
    chunks.append(chunk)
df_all = pd.concat(chunks)

In [ ]:
chrom = df_all[['gene', 'chr']].drop_duplicates()
chrom.index = chrom['gene']

In [ ]:
# load in chr/gene order 
chr_order = pd.read_csv('/home/cporter/atlas_remake_Aug_26_2024/docs/Trinity_CTAT_cnv_hg38_gencode_v27.txt', 
                        sep='\t', usecols=[0, 1, 2, 3], header=None) #update
chr_order.columns=['gene', 'chr', 'start', 'end']
chr_order

In [ ]:
# Make a cell_group_name x gene/chr matrix 
df_wide = df_all.pivot(index='cell_group_name', columns='gene', values='state')
genes_order = df_all['gene'].drop_duplicates()
df_wide = df_wide[genes_order]
df_wide.fillna(3, inplace=True) # replacing with 3 becuase in the HMM values, a value of 3 is "white" - do deletion or duplication 
df_wide
metadata = df_all[['cell_group_name', 'Decade', 'Cohort', 'MSI', 'Side', 'Stage', 'Sex', 'Treatment', 'Age', 'Mean_Counts', 'Total_Epi']].drop_duplicates()
df_wide = df_wide.merge(metadata, left_index=True, right_on='cell_group_name', how='left')
df_wide.index = df_wide['cell_group_name']

In [ ]:
gene_exists = pd.Index(chr_order['gene']).intersection(df_wide.columns)
df_wide_sorted = df_wide[gene_exists]

In [ ]:
# create vector with the CNV cluster number 
cnv_clusters = df_wide_sorted.index
s_label = cnv_clusters.str.split('_')

In [ ]:
df_wide_orig = df_wide
df_wide_sorted_orig = df_wide_sorted

# MSS Only

In [ ]:
df_wide = df_wide_orig[df_wide_orig['MSI'] == 'MSS: STABLE']
df_wide_sorted = df_wide_sorted_orig[df_wide_orig['MSI']=='MSS: STABLE']

In [ ]:
import pickle
with open(os.path.join(base_path, 'results/2025-10-01_YOCRC_inferCNV/MSS_dataframes_YOCRC.pkl'), "wb") as f:
    pickle.dump({
        "df_wide": df_wide,
        "df_wide_sorted": df_wide_sorted
    }, f)

In [ ]:
# Get stats for legend
df_new = pd.DataFrame({
    "FRID": df_wide["Cohort"].astype(str) + "_" + df_wide['cell_group_name'].str.split('_').str[0],
    "Cohort": df_wide["Cohort"]})

df_new.groupby('Cohort').agg(
    rows=('Cohort', 'count'),
    unique_FRIDs=('FRID', 'nunique')
)

In [ ]:
import seaborn as sns
import matplotlib.colors as mcolors
from scipy.spatial import distance
from scipy.cluster import hierarchy
from scipy.cluster.hierarchy import fcluster

row_dist = distance.pdist(df_wide_sorted, metric = 'correlation')
row_linkage = hierarchy.linkage(row_dist, method='ward', metric='correlation')

clusters = fcluster(row_linkage, t=2.5, criterion='distance')  # 2.8 for young, 3.8 for old
clusters
df_clusters = pd.DataFrame({'sample': df_wide_sorted.index, 'cluster': clusters})
df_clusters.sort_values(by = "cluster", inplace = True)

hex_colors = ["#F55D4F", '#E69F00', '#56B4E9', '#009E73', '#F0E442', '#0072B2', '#D55E00', '#CC79A7', 
           "#F1A7C3", "#E8C55C", "#0B6E4F", "#84B8F7", "#A1D47E", "#E9B2A6", "#C6D4E2", "#F9D79C", 
           "#A9D0F5", "#77A1D4", "#D1A1F1", "#8C1A61", "#C0F2A1", "#E19F67", "#2AAB62", "#F1A943", 
           "#BB74D2", "#65D7D2", "#A76F8B", "#C9A08A", "#63B2B3", "#F4A7BC", "#5A6261", 
           "#FFD16C", "#70B4AE", "#34A8A1", "#E4C697", "#B9C6D5", "#D4A358", "#4F8D7C", "#BEF1B6"]
lut = dict(zip(df_clusters['cluster'].unique(), hex_colors[18:len(hex_colors)]))
cluster_colors = df_clusters['cluster'].map(lut)
cluster_colors.index = df_clusters['sample']

row_annotation = df_wide[['Cohort', 'Side', 'Decade', 'Stage']]
cohort_colors = df_wide['Cohort'].map({'UnderFifty': '#56B4E9', 'FiftyPlus': '#E69F00'})
decade_colors = df_wide['Decade'].map({'20-29': hex_colors[0], '30-39': hex_colors[1], '40-49': hex_colors[2],
                                      '50-59': hex_colors[3], '60-69': hex_colors[4], '70-79': hex_colors[5], 
                                      '80-89': hex_colors[6], '90-99': hex_colors[7]})

side_colors = df_wide['Side'].map({'Left': '#009E73', 'Right': '#F0E442', 'Rectal': '#CC79A7'})  # Adjust as per your Side categories
stage_colors = df_wide['Stage'].map({'I': hex_colors[0], 'II': hex_colors[1], 'III': hex_colors[2], 'IV': hex_colors[3]})
row_colors_all = pd.DataFrame({'Decade': decade_colors, 'Cohort': cohort_colors, 'Side': side_colors, 
                               'Stage': stage_colors, 'Cluster': cluster_colors})
row_colors = pd.DataFrame({'Cohort': cohort_colors, 'Cluster': cluster_colors})

col_annotation = chrom['chr']
col_colors = col_annotation.map({'chr1': 'Crimson', 
                                 'chr2': 'SlateBlue', 
                                 'chr3': 'ForestGreen', 
                                 'chr4': 'DodgerBlue', 
                                 'chr5': 'MediumPurple', 
                                 'chr6': 'Tomato', 
                                 'chr7': 'Goldenrod', 
                                 'chr8': 'Coral', 
                                 'chr9': 'OliveDrab', 
                                 'chr10': 'DarkOrange', 
                                 'chr11': 'MediumSeaGreen', 
                                 'chr12': 'SteelBlue', 
                                 'chr13': 'RoyalBlue', 
                                 'chr14': 'Indigo', 
                                 'chr15': 'LimeGreen', 
                                 'chr16': 'PapayaWhip', 
                                 'chr17': 'Turquoise', 
                                 'chr18': 'Chocolate', 
                                 'chr19': 'FireBrick', 
                                 'chr20': 'HotPink', 
                                 'chr21': 'LavenderBlush', 
                                 'chr22': 'SeaGreen', 
                                 'chrM': 'SaddleBrown', 
                                 'chrX': 'MediumVioletRed', 
                                 'chrY': 'SkyBlue'})

cmap = mcolors.LinearSegmentedColormap.from_list("blue_white_red", ["blue", "white", "red"])
norm = mcolors.TwoSlopeNorm(vmin=df_wide_sorted.min().min(), vcenter=3, vmax=df_wide_sorted.max().max())

ax = sns.clustermap(df_wide_sorted, 
                    row_linkage=row_linkage, 
                    #row_cluster=True, 
                    col_cluster=False,
                    row_colors=row_colors, 
                    col_colors=col_colors,
                    cmap=cmap, 
                    norm=norm, 
                    figsize=(25,df_wide.shape[0]/10))

ax.ax_heatmap.set_xticklabels([])  # Remove x-axis tick labels
ax.ax_heatmap.set_yticklabels([])
ax.ax_heatmap.set_xlabel("Gene/Chromosome")
ax.ax_heatmap.set_ylabel("SampleID-InferCNV-Cluster")
ax.ax_heatmap.grid(False)

plt.show()
ax.savefig(os.path.join(base_path, 'results/DATE_EOCRC_inferCNV/figures/allSamples_HMM_clustermap_MSS.pdf'), dpi=600)

In [ ]:
# look at cluster/color mapping 
lut

In [ ]:
row_order = ax.dendrogram_row.reordered_ind
sorted_sample_ids = df_wide_sorted.index[row_order]

df_cluster_export = pd.DataFrame({
    "sample_id": sorted_sample_ids,
    "cluster": clusters[row_order]
})

df_cluster_export['stage'] = df_cluster_export['sample_id'].map(df_wide['Stage'])
df_cluster_export['sex']   = df_cluster_export['sample_id'].map(df_wide['Sex'])
df_cluster_export['side']  = df_cluster_export['sample_id'].map(df_wide['Side'])
df_cluster_export['age']   = df_cluster_export['sample_id'].map(df_wide['Age'])
df_cluster_export['treat'] = df_cluster_export['sample_id'].map(df_wide['Treatment'])
df_cluster_export['cohort'] = df_cluster_export['sample_id'].map(df_wide['Cohort'])
df_cluster_export['decade'] = df_cluster_export['sample_id'].map(df_wide['Decade'])
df_cluster_export['counts'] = df_cluster_export['sample_id'].map(df_wide['Mean_Counts'])
df_cluster_export['epi_counts'] = df_cluster_export['sample_id'].map(df_wide['Total_Epi'])

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

df_cluster_export['patient_id'] = df_cluster_export['sample_id'].str.rsplit('_T', n=1).str[0]

df_cluster_export.head()

In [ ]:
# performed age association with clusters in a seperate R script to use mixed effect model 
df_cluster_export.to_csv(os.path.join(base_path, 'results/EOCRC_inferCNV/CNV_cluster_df.csv'), index=False)

# assessing number of CNV subsets by age 

In [ ]:
df_cluster_export['subcluster_id'] = df_cluster_export['sample_id'].str.split('_').str[-1]
df_cluster_export['patient_id'] = df_cluster_export['sample_id'].str.rsplit('_T', n=1).str[0]
df_cluster_export['s_num'] = df_cluster_export['subcluster_id'].str.replace('s', '').astype(int)
patient_df = df_cluster_export.groupby('patient_id').agg({
    's_num': 'max',            # Total number of subclusters
    'age_scaled': 'first',     # Age stays the same for the patient
    'side': 'first',
    'treat': 'first',
    'stage': 'first',
    'sex': 'first', 
    'cohort': 'first',
    'age': 'first',
    'decade': 'first', 
    'counts': 'first', 
    'epi_counts': 'first', 
}).reset_index()

patient_df = patient_df.rename(columns={'s_num': 'total_subclusters'})
patient_df.head()

In [ ]:
import statsmodels.formula.api as smf

print("testing >=2...")
patient_df['high_complexity'] = (patient_df['total_subclusters'] >= 2).astype(int)
logit_model = smf.logit('high_complexity ~ age_scaled + side + treat + stage + sex + counts + epi_counts', data=patient_df).fit()
print(logit_model.summary())

print("testing >=3...")
patient_df['high_complexity'] = (patient_df['total_subclusters'] >= 3).astype(int)
logit_model = smf.logit('high_complexity ~ age_scaled + side + treat + stage + sex + counts + epi_counts', data=patient_df).fit()
print(logit_model.summary())

In [ ]:
plt.figure(figsize=(6, 6))

sns.boxplot(data=patient_df, x='total_subclusters', y='age', 
            width=0.4, fliersize=0,
            palette=["#F55D4F", '#E69F00', '#56B4E9', '#009E73'], 
            boxprops={'edgecolor':'gray', 'alpha':0.9})

sns.stripplot(data=patient_df, x='total_subclusters', y='age', 
              hue='total_subclusters', color='black',
              size=6, alpha=0.5, jitter=0.2)

plt.xlabel('# of InferCNV subsets per tumor', fontsize=12)
plt.ylabel('Age', fontsize=12)
plt.grid(axis='y', alpha=0.3)
plt.legend().remove() 

plt.savefig(os.path.join(base_path, 'results/DATE_EOCRC_inferCNV/CNV_subsets_per_tumor_MSS.pdf'), bbox_inches='tight')
plt.show()

# MSI

In [ ]:
df_wide = df_wide_orig[df_wide_orig['MSI'] == 'MSI-H: HIGH']
df_wide_sorted = df_wide_sorted_orig[df_wide_orig['MSI']=='MSI-H: HIGH']

In [ ]:
# Get stats for legend
df_new = pd.DataFrame({
    "FRID": df_wide["Cohort"].astype(str) + "_" + df_wide['cell_group_name'].str.split('_').str[0],
    "Cohort": df_wide["Cohort"]})

df_new.groupby('Cohort').agg(
    rows=('Cohort', 'count'),
    unique_FRIDs=('FRID', 'nunique')
)

In [ ]:
import seaborn as sns
import matplotlib.colors as mcolors
from scipy.spatial import distance
from scipy.cluster import hierarchy
from scipy.cluster.hierarchy import fcluster

row_dist = distance.pdist(df_wide_sorted, metric = 'correlation')
row_linkage = hierarchy.linkage(row_dist, method='ward', metric='correlation')

clusters = fcluster(row_linkage, t=2.5, criterion='distance')  # 2.8 for young, 3.8 for old
clusters
df_clusters = pd.DataFrame({'sample': df_wide_sorted.index, 'cluster': clusters})
df_clusters.sort_values(by = "cluster", inplace = True)

hex_colors = ["#F55D4F", '#E69F00', '#56B4E9', '#009E73', '#F0E442', '#0072B2', '#D55E00', '#CC79A7', 
           "#F1A7C3", "#E8C55C", "#0B6E4F", "#84B8F7", "#A1D47E", "#E9B2A6", "#C6D4E2", "#F9D79C", 
           "#A9D0F5", "#77A1D4", "#D1A1F1", "#8C1A61", "#C0F2A1", "#E19F67", "#2AAB62", "#F1A943", 
           "#BB74D2", "#65D7D2", "#A76F8B", "#C9A08A", "#63B2B3", "#F4A7BC", "#5A6261", 
           "#FFD16C", "#70B4AE", "#34A8A1", "#E4C697", "#B9C6D5", "#D4A358", "#4F8D7C", "#BEF1B6"]
lut = dict(zip(df_clusters['cluster'].unique(), hex_colors[18:len(hex_colors)]))
cluster_colors = df_clusters['cluster'].map(lut)
cluster_colors.index = df_clusters['sample']

row_annotation = df_wide[['Cohort', 'Side', 'Decade', 'Stage']]
cohort_colors = df_wide['Cohort'].map({'UnderFifty': '#56B4E9', 'FiftyPlus': '#E69F00'})
decade_colors = df_wide['Decade'].map({'20-29': hex_colors[0], '30-39': hex_colors[1], '40-49': hex_colors[2],
                                      '50-59': hex_colors[3], '60-69': hex_colors[4], '70-79': hex_colors[5], 
                                      '80-89': hex_colors[6], '90-99': hex_colors[7]})

side_colors = df_wide['Side'].map({'Left': '#009E73', 'Right': '#F0E442', 'Rectal': '#CC79A7'})  # Adjust as per your Side categories
stage_colors = df_wide['Stage'].map({'I': hex_colors[0], 'II': hex_colors[1], 'III': hex_colors[2], 'IV': hex_colors[3]})
row_colors_all = pd.DataFrame({'Decade': decade_colors, 'Cohort': cohort_colors, 'Side': side_colors, 
                               'Stage': stage_colors, 'Cluster': cluster_colors})
row_colors = pd.DataFrame({'Cohort': cohort_colors, 'Cluster': cluster_colors})

col_annotation = chrom['chr']
col_colors = col_annotation.map({'chr1': 'Crimson', 
                                 'chr2': 'SlateBlue', 
                                 'chr3': 'ForestGreen', 
                                 'chr4': 'DodgerBlue', 
                                 'chr5': 'MediumPurple', 
                                 'chr6': 'Tomato', 
                                 'chr7': 'Goldenrod', 
                                 'chr8': 'Coral', 
                                 'chr9': 'OliveDrab', 
                                 'chr10': 'DarkOrange', 
                                 'chr11': 'MediumSeaGreen', 
                                 'chr12': 'SteelBlue', 
                                 'chr13': 'RoyalBlue', 
                                 'chr14': 'Indigo', 
                                 'chr15': 'LimeGreen', 
                                 'chr16': 'PapayaWhip', 
                                 'chr17': 'Turquoise', 
                                 'chr18': 'Chocolate', 
                                 'chr19': 'FireBrick', 
                                 'chr20': 'HotPink', 
                                 'chr21': 'LavenderBlush', 
                                 'chr22': 'SeaGreen', 
                                 'chrM': 'SaddleBrown', 
                                 'chrX': 'MediumVioletRed', 
                                 'chrY': 'SkyBlue'})

cmap = mcolors.LinearSegmentedColormap.from_list("blue_white_red", ["blue", "white", "red"])
norm = mcolors.TwoSlopeNorm(vmin=df_wide_sorted.min().min(), vcenter=3, vmax=df_wide_sorted.max().max())

ax = sns.clustermap(df_wide_sorted, 
                    row_linkage=row_linkage, 
                    #row_cluster=True, 
                    col_cluster=False,
                    row_colors=row_colors, 
                    col_colors=col_colors,
                    cmap=cmap, 
                    norm=norm, 
                    figsize=(25,df_wide.shape[0]/10))

ax.ax_heatmap.set_xticklabels([])  # Remove x-axis tick labels
ax.ax_heatmap.set_yticklabels([])
ax.ax_heatmap.set_xlabel("Gene/Chromosome")
ax.ax_heatmap.set_ylabel("SampleID-InferCNV-Cluster")
ax.ax_heatmap.grid(False)

plt.show()
ax.savefig(os.path.join(base_path, 'results/DATE_EOCRC_inferCNV/figures/allSamples_HMM_clustermap_MSI.pdf'), dpi=600)

In [ ]:
row_order = ax.dendrogram_row.reordered_ind
sorted_sample_ids = df_wide_sorted.index[row_order]

df_cluster_export = pd.DataFrame({
    "sample_id": sorted_sample_ids,
    "cluster": clusters[row_order]
})

df_cluster_export['stage'] = df_cluster_export['sample_id'].map(df_wide['Stage'])
df_cluster_export['sex']   = df_cluster_export['sample_id'].map(df_wide['Sex'])
df_cluster_export['side']  = df_cluster_export['sample_id'].map(df_wide['Side'])
df_cluster_export['age']   = df_cluster_export['sample_id'].map(df_wide['Age'])
df_cluster_export['treat'] = df_cluster_export['sample_id'].map(df_wide['Treatment'])
df_cluster_export['cohort'] = df_cluster_export['sample_id'].map(df_wide['Cohort'])
df_cluster_export['decade'] = df_cluster_export['sample_id'].map(df_wide['Decade'])
df_cluster_export['counts'] = df_cluster_export['sample_id'].map(df_wide['Mean_Counts'])
df_cluster_export['epi_counts'] = df_cluster_export['sample_id'].map(df_wide['Total_Epi'])

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
df_cluster_export.head()

In [ ]:
df_cluster_export['subcluster_id'] = df_cluster_export['sample_id'].str.split('_').str[-1]
df_cluster_export['patient_id'] = df_cluster_export['sample_id'].str.rsplit('_T', n=1).str[0]
df_cluster_export['s_num'] = df_cluster_export['subcluster_id'].str.replace('s', '').astype(int)
patient_df = df_cluster_export.groupby('patient_id').agg({
    's_num': 'max',            # Total number of subclusters
    'age_scaled': 'first',     # Age stays the same for the patient
    'side': 'first',
    'treat': 'first',
    'stage': 'first',
    'sex': 'first', 
    'cohort': 'first',
    'age': 'first',
    'counts': 'first', 
    'epi_counts': 'first'
    
}).reset_index()

patient_df = patient_df.rename(columns={'s_num': 'total_subclusters'})
patient_df.head()

In [ ]:
import statsmodels.formula.api as smf

print("testing >=2...")
patient_df['high_complexity'] = (patient_df['total_subclusters'] >= 2).astype(int)
logit_model = smf.logit('high_complexity ~ age_scaled + side + treat + stage + sex + counts + epi_counts', data=patient_df).fit()
print(logit_model.summary())

print("testing >=3...")
patient_df['high_complexity'] = (patient_df['total_subclusters'] >= 3).astype(int)
logit_model = smf.logit('high_complexity ~ age_scaled + side + treat + stage + sex + counts + epi_counts', data=patient_df).fit()
print(logit_model.summary())

In [ ]:
plt.figure(figsize=(6, 6))

sns.boxplot(data=patient_df, x='total_subclusters', y='age', 
            width=0.4, fliersize=0,
            palette=["#F55D4F", '#E69F00', '#56B4E9', '#009E73'], 
            boxprops={'edgecolor':'gray', 'alpha':0.9})

sns.stripplot(data=patient_df, x='total_subclusters', y='age', 
              hue='total_subclusters', color='black',
              size=6, alpha=0.5, jitter=0.2)

plt.xlabel('# of InferCNV subsets per tumor', fontsize=12)
plt.ylabel('Age', fontsize=12)
plt.grid(axis='y', alpha=0.3)
plt.legend().remove() 

plt.savefig(os.path.join(base_path, 'results/DATE_EOCRC_inferCNV/CNV_subsets_per_tumor_MSI.pdf'), bbox_inches='tight')
plt.show()

# CNV burden by cell type and age 

In [ ]:
adata = sc.read_h5ad(os.path.join(base_path, 'data/yocrc_Epithelial_annotation_noHarmony.h5ad')) # no harmony 

In [ ]:
# add residuals to adata
cnvs = pd.read_csv(os.path.join(base_path, 'results/DATE_EOCRC_inferCNV/metadata-sum_abslog2_residuals.tsv'), sep = '\t', index_col=None)
cnvs.set_index('barcode', inplace=True)

In [ ]:
cnvs[["sum_abslog2_resids"]].isna().sum()

In [ ]:
adata.obs['cvns_sum_abslog2_resids']=cnvs

In [ ]:
# MSS 
sc.set_figure_params(figsize=(4, 4))
fig, ax = plt.subplots()
sc.pl.violin(
    adata[
    (adata.obs['MSI_v2'] == "MSS: STABLE") &
    (~adata.obs['Annotation_Tier2'].isin(["Mixed - epithelial", "Patient-specific"]))
    ],
    'cvns_sum_abslog2_resids',
    groupby='Annotation_Tier2',
    log=True,
    use_raw=True,
    rotation=90,
    stripplot=False,  # hides the dots (stripplot)
    inner='quartile',  # shows median and quartile lines
    #density_norm='width',
    ax = ax)

fig.savefig(os.path.join(base_path, 'results/DATE_YOCRC_inferCNV/figures/abslog2cnv_byTier2_epi_celltype_MSS.pdf'), dpi=600, bbox_inches='tight')
plt.close(fig)

In [ ]:
obs_df = adata.obs.copy()

mask = (obs_df['MSI_v2'] == "MSS: STABLE") & \
       (~obs_df['Annotation_Tier2'].isin(["Mixed - epithelial", "Patient-specific"]))

filtered_obs = obs_df[mask]

tumor_means = filtered_obs.groupby('FRID').agg({
    'cvns_sum_abslog2_resids': 'mean',
    'Age': 'first',
    'Overall_Stage': 'first',
    'Sex': 'first',
    'Therapy_v2': 'first',
    'Sidedness': 'first'
})

tumor_means = tumor_means.rename(columns={'cvns_sum_abslog2_resids': 'mean_abslog2cnv'})

tumor_means['Age'] = pd.to_numeric(tumor_means['Age'], errors='coerce')
tumor_means.head()

In [ ]:
import statsmodels.formula.api as smf
tumor_means['Age'] = pd.to_numeric(tumor_means['Age'], errors='coerce')
tumor_means['age_scaled'] = (tumor_means['Age'] - tumor_means['Age'].mean()) / tumor_means['Age'].std()

model = smf.ols('mean_abslog2cnv ~ age_scaled + Sidedness + Overall_Stage + Sex + Therapy_v2', 
                data=tumor_means).fit()

print(model.summary())

In [ ]:
# MSI 
sc.set_figure_params(figsize=(4, 4))
fig, ax = plt.subplots()

# 2. Force Python to delete the ghost memory of those empty categories
adata_subset.obs['Annotation_Tier2'] = adata_subset.obs['Annotation_Tier2'].cat.remove_unused_categories()
sc.pl.violin(
    adata[
    (adata.obs['MSI_v2'] == "MSI-H: HIGH") &
    (~adata.obs['Annotation_Tier2'].isin(["Mixed - epithelial", "Patient-specific"]))
    ],
    keys='cvns_sum_abslog2_resids',
    groupby='Annotation_Tier2',
    log=False, # Let matplotlib handle the log transformation instead
    use_raw=True,
    rotation=90,
    stripplot=False,
    inner='quartile',
    ax=ax,
    show=False, 
    scale='width',
)
ax.set_yscale('log')
plt.show()
fig.savefig(os.path.join(base_path, 'results/DATE_EOCRC_inferCNV/figures/abslog2cnv_byTier2_epi_celltype_MSI.pdf'), dpi=600, bbox_inches='tight')
plt.close(fig)

In [ ]:
obs_df = adata.obs.copy()

mask = (obs_df['MSI_v2'] == "MSI-H: HIGH") & \
       (~obs_df['Annotation_Tier2'].isin(["Mixed - epithelial", "Patient-specific"]))

filtered_obs = obs_df[mask]

tumor_means = filtered_obs.groupby('FRID').agg({
    'cvns_sum_abslog2_resids': 'mean',
    'Age': 'first',
    'Overall_Stage': 'first',
    'Sex': 'first',
    'Therapy_v2': 'first',
    'Sidedness': 'first'
})

tumor_means = tumor_means.rename(columns={'cvns_sum_abslog2_resids': 'mean_abslog2cnv'})

tumor_means['Age'] = pd.to_numeric(tumor_means['Age'], errors='coerce')
tumor_means.head()

In [ ]:
import statsmodels.formula.api as smf
tumor_means['Age'] = pd.to_numeric(tumor_means['Age'], errors='coerce')
tumor_means['age_scaled'] = (tumor_means['Age'] - tumor_means['Age'].mean()) / tumor_means['Age'].std()

model = smf.ols('mean_abslog2cnv ~ age_scaled + Sidedness + Overall_Stage + Sex', 
                data=tumor_means).fit()

print(model.summary())